# SAM2 Biceps Femoris Segmentation

Prototype replacing SAM1 box-per-slice with **SAM2 video propagation**.

Key difference from `biceps_pipeline.py`:
- The detection pipeline (`crotch_cut` / `leg_boxes`) only needs to succeed on **one seed slice** in the middle of the volume.
- SAM2 propagates that mask forward and backward through the slice stack, covering the end-of-volume slices that currently fail.
- Masks are spatially consistent across slices rather than independently computed.

In [ ]:
import shutil
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

# Reuse loading + detection helpers from the existing pipeline
from biceps_pipeline import (
    bone_mask,
    crotch_cut,
    foreground_mask,
    leg_boxes,
    load_dicom_images,
)

from sam2.build_sam import build_sam2_video_predictor

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path.cwd()

DICOM_ROOT = PROJECT_ROOT / "DICOM_Files"
OUT_DIR = PROJECT_ROOT / "masks_out"
SAM2_CKPT = PROJECT_ROOT / "sam2.1_hiera_large.pt"
SAM2_CFG = "configs/sam2.1/sam2.1_hiera_l.yaml"

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
assert SAM2_CKPT.exists(), f"Checkpoint not found: {SAM2_CKPT}"

In [ ]:
# Build once, reuse for every series
predictor = build_sam2_video_predictor(SAM2_CFG, str(SAM2_CKPT), device=DEVICE)
print("SAM2 predictor ready")

## Helper functions

In [ ]:
def find_seed_slice(imgs_echo, *, search_start=15, search_end=36):
    """Scan the middle of the volume for the first slice where the full
    detection pipeline succeeds.  Returns (slice_index, boxes) or (None, None).
    
    We only need ONE clean slice — SAM2 propagates the rest.
    """
    for s in range(search_start, search_end):
        mag = imgs_echo[s]
        fg = foreground_mask(mag)
        bone = bone_mask(mag)
        cut = crotch_cut(fg, bone)
        if cut is None:
            continue
        masked = mag * (fg & ~bone) * cut
        boxes = leg_boxes(masked, bone)
        if boxes is not None:
            return s, boxes
    return None, None

In [ ]:
def write_frames(imgs_echo, tmp_dir):
    """Save the echo slice stack as JPEGs for SAM2's video predictor.

    Normalises the *whole volume* once so brightness is consistent across
    frames — important for SAM2's temporal features to work correctly.
    (Per-slice normalisation like SAM1 used would make slices look like
    unrelated images and break propagation.)

    SAM2 requires JPEG files specifically -- its loader ignores other formats.
    """
    mn, mx = imgs_echo.min(), imgs_echo.max()
    vol_norm = (imgs_echo - mn) / (mx - mn) if mx > mn else imgs_echo

    tmp = Path(tmp_dir)
    for s in range(imgs_echo.shape[0]):
        u8 = (vol_norm[s] * 255).astype(np.uint8)
        rgb = np.stack([u8, u8, u8], axis=-1)
        Image.fromarray(rgb).save(tmp / f"{s:05d}.jpg", quality=95)

In [ ]:
from scipy.ndimage import gaussian_filter


def smooth_mask_volume(mask_volume, *, sigma_z=1.5, sigma_xy=1.0, threshold=0.5):
    """Smooth the binary mask volume using an anisotropic 3D Gaussian.

    Each slice is informed by its neighbours along the z-axis, so borders
    that jump between back-to-back frames are softened.  Re-thresholding
    at 0.5 keeps the result binary.

    sigma_z   -- smoothing strength along the slice axis (inter-slice).
                 1.5 means ~3 slices contribute to each pixel decision.
    sigma_xy  -- smoothing within each slice.  Keep small (0.5-1.0) to
                 avoid eroding fine in-plane boundary detail.
    threshold -- re-binarisation cutoff after blurring.
    """
    blurred = gaussian_filter(
        mask_volume.astype(np.float32),
        sigma=(sigma_z, sigma_xy, sigma_xy),
    )
    return blurred > threshold


In [ ]:
import matplotlib.patches as patches


def debug_slice(folder, slice_idx, *, echo=0, dark_thresh=0.23, circularity_min=0.5):
    """Show each intermediate mask for one slice.

    Panels: original | foreground | bone | after crotch cut | bounding boxes.
    Useful for diagnosing why find_seed_slice fails or to inspect seed quality.

    Example:
        debug_slice(TEST_FOLDER, 15)
        debug_slice(TEST_FOLDER, 3)   # likely fails at crotch_cut — shows where
    """
    imgs = load_dicom_images(folder)
    mag = imgs[echo, slice_idx]

    fg = foreground_mask(mag)
    bone = bone_mask(mag, dark_thresh=dark_thresh, circularity_min=circularity_min)
    cut = crotch_cut(fg, bone)

    if cut is not None:
        masked = mag * (fg & ~bone) * cut
        boxes = leg_boxes(masked, bone)
    else:
        masked = None
        boxes = None

    fig, axes = plt.subplots(1, 5, figsize=(18, 4))

    axes[0].imshow(mag, cmap="gray", vmin=0, vmax=1)
    axes[0].set_title("Original")

    axes[1].imshow(fg, cmap="gray")
    axes[1].set_title("Foreground")

    axes[2].imshow(bone, cmap="gray")
    axes[2].set_title("Bone")

    if masked is not None:
        axes[3].imshow(masked, cmap="gray", vmin=0, vmax=1)
        axes[3].set_title("Crotch cut")
    else:
        axes[3].imshow(mag, cmap="gray", vmin=0, vmax=1, alpha=0.4)
        axes[3].set_title("Crotch cut\n(failed)", color="red")

    if masked is not None:
        axes[4].imshow(masked, cmap="gray", vmin=0, vmax=1)
        if boxes is not None:
            for box in boxes:
                x0, y0, x1, y1 = box
                axes[4].add_patch(patches.Rectangle(
                    (x0, y0), x1 - x0, y1 - y0,
                    linewidth=1.5, edgecolor="red", facecolor="none",
                ))
            axes[4].set_title("Boxes")
        else:
            axes[4].set_title("Boxes\n(failed)", color="red")
    else:
        axes[4].imshow(np.zeros_like(mag), cmap="gray")
        axes[4].set_title("Boxes\n(failed)", color="red")

    for ax in axes:
        ax.axis("off")

    fig.suptitle(
        f"{Path(folder).name}  —  echo {echo}, slice {slice_idx}",
        fontsize=11,
    )
    plt.tight_layout()
    plt.show()

## Core segmentation function

In [ ]:
def _bone_centroid_in_box(bone_2d, box):
    x1, y1, x2, y2 = box
    region = bone_2d[y1:y2 + 1, x1:x2 + 1]
    if not region.any():
        return None
    ys, xs = np.where(region)
    return float(xs.mean() + x1), float(ys.mean() + y1)


def _find_visible_range(confidence, seed_idx, conf_threshold=0.25):
    """Walk outward from seed_idx and stop the first time peak SAM2 logit
    drops below conf_threshold * seed confidence.  Returns (first_vis, last_vis).

    Walking outward rather than global thresholding ensures a single noisy
    middle slice cannot truncate the visible range prematurely.
    """
    seed_conf = confidence[seed_idx]
    cutoff = seed_conf * conf_threshold

    first_vis = seed_idx
    while first_vis > 0 and confidence[first_vis - 1] >= cutoff:
        first_vis -= 1

    last_vis = seed_idx
    while last_vis < len(confidence) - 1 and confidence[last_vis + 1] >= cutoff:
        last_vis += 1

    return first_vis, last_vis

In [ ]:
def segment_series_sam2(
        folder,
        predictor,
        *,
        echo=0,
        n_echo=7,
        n_slices=50,
        shape=(192, 192),
        seed_search=(15, 36),
        slice_range=None,
        reverse_slices=False,  # flip the slice stack before processing
        dark_thresh=0.23,
        circularity_min=0.5,
        conf_threshold=0.25,
):
    H, W = shape
    imgs = load_dicom_images(folder, n_echo=n_echo, n_slices_per_echo=n_slices, shape=shape)
    imgs_echo = imgs[echo]
    if reverse_slices:
        imgs_echo = imgs_echo[::-1].copy()

    # Resolve valid window (range_end is exclusive internally)
    range_start = 0 if slice_range is None else slice_range[0]
    range_end = n_slices if slice_range is None else slice_range[1] + 1
    imgs_valid = imgs_echo[range_start:range_end]  # only these frames go to SAM2
    n_valid = range_end - range_start
    print(f"  Valid slice range: {range_start}–{range_end - 1} ({n_valid} slices)")

    # 1. Find seed inside valid window (indices are local to imgs_valid)
    ss_start = max(0, seed_search[0] - range_start)
    ss_end = min(n_valid, seed_search[1] - range_start)
    seed_local, boxes = find_seed_slice(imgs_valid, search_start=ss_start, search_end=ss_end)
    if seed_local is None:
        raise RuntimeError(
            f"No clean seed slice found in search window "
            f"[{range_start + ss_start}, {range_start + ss_end}) (absolute).  "
            "Try widening seed_search or adjusting dark_thresh / circularity_min."
        )
    seed_idx = seed_local + range_start
    print(f"  Seed slice: {seed_idx} (local frame {seed_local})  |  boxes: {boxes}")

    seed_bone = bone_mask(imgs_valid[seed_local], dark_thresh=dark_thresh, circularity_min=circularity_min)
    neg_points = [_bone_centroid_in_box(seed_bone, box) for box in boxes]
    print(f"  Negative points (bone centroids): {neg_points}")

    # 2. Write only valid frames; SAM2 frame indices are local [0, n_valid)
    tmp_dir = tempfile.mkdtemp(prefix="sam2_mri_")
    try:
        write_frames(imgs_valid, tmp_dir)

        # 3. Run SAM2 — all frame_idx values are local to the valid window
        mask_volume = np.zeros((n_slices, H, W), dtype=bool)
        confidence = np.zeros(n_valid, dtype=np.float32)

        with torch.inference_mode(), torch.autocast(DEVICE, dtype=torch.bfloat16):
            state = predictor.init_state(video_path=tmp_dir)

            for obj_id, (box, neg_pt) in enumerate(zip(boxes, neg_points), start=1):
                if neg_pt is not None:
                    predictor.add_new_points_or_box(
                        inference_state=state,
                        frame_idx=seed_local,
                        obj_id=obj_id,
                        box=np.array(box, dtype=np.float32),
                        points=np.array([[neg_pt[0], neg_pt[1]]], dtype=np.float32),
                        labels=np.array([0], dtype=np.int32),
                    )
                else:
                    predictor.add_new_points_or_box(
                        inference_state=state,
                        frame_idx=seed_local,
                        obj_id=obj_id,
                        box=np.array(box, dtype=np.float32),
                    )

            fwd_masks, fwd_conf = {}, {}
            for fi, _oids, logits in predictor.propagate_in_video(
                    state, start_frame_idx=seed_local
            ):
                fwd_masks[fi] = (logits[:, 0] > 0).any(dim=0).cpu().numpy()
                fwd_conf[fi] = float(logits.max().cpu())

            bwd_masks, bwd_conf = {}, {}
            for fi, _oids, logits in predictor.propagate_in_video(
                    state, start_frame_idx=seed_local, reverse=True
            ):
                bwd_masks[fi] = (logits[:, 0] > 0).any(dim=0).cpu().numpy()
                bwd_conf[fi] = float(logits.max().cpu())

            predictor.reset_state(state)

        # Map local SAM2 frame index → global slice index in mask_volume
        for fi in range(n_valid):
            si = fi + range_start
            if fi in fwd_masks:
                mask_volume[si] = fwd_masks[fi]
                confidence[fi] = fwd_conf[fi]
            elif fi in bwd_masks:
                mask_volume[si] = bwd_masks[fi]
                confidence[fi] = bwd_conf[fi]

    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)

    # 4. Trim to where muscle is actually visible (local indices → convert to global)
    first_local, last_local = _find_visible_range(confidence, seed_local, conf_threshold)
    first_vis = first_local + range_start
    last_vis = last_local + range_start
    mask_volume[:first_vis] = False
    mask_volume[last_vis + 1:] = False
    print(f"  Visible range: slices {first_vis}–{last_vis}  "
          f"(trimmed {first_vis} proximal + {n_slices - 1 - last_vis} distal)")

    # 5. Subtract bone mask within visible range
    for s in range(first_vis, last_vis + 1):
        if mask_volume[s].any():
            b = bone_mask(imgs_echo[s], dark_thresh=dark_thresh, circularity_min=circularity_min)
            mask_volume[s] = mask_volume[s] & ~b

    # 6. Smooth within valid window only
    valid_region = mask_volume[range_start:range_end]
    mask_volume[range_start:range_end] = smooth_mask_volume(valid_region)

    # 7. Broadcast to all echoes
    mask_all_echoes = np.broadcast_to(mask_volume, (n_echo, n_slices, H, W)).copy()

    n_covered = int(mask_volume.any(axis=(1, 2)).sum())
    report = {
        "seed_slice": seed_idx,
        "first_vis": first_vis,
        "last_vis": last_vis,
        "range_start": range_start,
        "range_end": range_end - 1,
        "n_covered": n_covered,
        "n_total": n_slices,
    }
    return mask_volume, mask_all_echoes, report

## Batch runner

In [ ]:
# ── Per-series valid slice ranges ─────────────────────────────────────────────
# Keys match the output filename stem: "{date}_{series_folder}".
# Values are (start, end) inclusive.  Set to None to use all 50 slices.
SLICE_RANGES = {
    "20240709_GRE2D_FATWATER_WAYLON_0012": None,
    "20240710_GRE2D_FATWATER_SUSHI_0012": (13, 25),
    "20240711-1_GRE2D_FATWATER_APHRODITE_0012": (5, 37),
    "20240711-2_GRE2D_FATWATER_SELENE_0012": (12, 43),
    "20240923_GRE2D_FATWATER_SUSHI_0012": (13, 32),
    "20240924_GRE2D_FATWATER_APHRODITE_0012": (8, 37),
    "20240925_GRE2D_FATWATER_WAYLON_0012": (10, 37),
    "20240926_GRE2D_FATWATER_SELENE_0012": (4, 30),
    "20250113_GRE2D_FATWATER_WAYLON_0012": (8, 42),
    "20250115_GRE2D_FATWATER_SELENE_0012": (9, 39),
    "20250127_GRE2D_FATWATER_APHRODITE_0012": (9, 27),
    "20250129_GRE2D_FATWATER_SUSHI_0012": (13, 34),
    "20250501_GRE2D_FATWATER_WAYLON_0012": (14, 49),
    "20250502_GRE2D_FATWATER_SELENE_0012": (18, 46),
    "20250505_GRE2D_FATWATER_APHRODITE_0012": (20, 44),
    "20250506_GRE2D_FATWATER_SUSHI_0012": (16, 35),
}

In [ ]:
def process_all_sam2(
        dicom_root,
        predictor,
        out_dir,
        *,
        pattern="*GRE2D_FATWATER*0012",
        slice_ranges=None,
):
    """Process every magnitude series and save (n_echo, 50, H, W) mask .npy files.

    slice_ranges : dict | None
        Maps each series key "{date}_{folder}" to its (start, end) inclusive
        slice range.  Missing keys and None values both mean "use all slices".
        Pass SLICE_RANGES from the config cell above.
    """
    dicom_root = Path(dicom_root)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    series_folders = sorted(p for p in dicom_root.rglob(pattern) if p.is_dir())
    print(f"Found {len(series_folders)} series")

    summary = []
    for folder in series_folders:
        name = f"{folder.parent.name}_{folder.name}"
        sr = (slice_ranges or {}).get(name)  # None → all slices
        print(f"\n{name}  (slice_range={sr})")
        try:
            _, mask_all, rep = segment_series_sam2(folder, predictor, slice_range=sr)
        except Exception as e:
            print(f"  ERROR: {e}")
            summary.append((name, f"ERROR: {e}"))
            continue

        np.save(out_dir / f"{name}_mask.npy", mask_all)
        msg = (
            f"{rep['n_covered']}/{rep['n_total']} slices covered  "
            f"(seed={rep['seed_slice']}, range={rep['range_start']}–{rep['range_end']})"
        )
        print(f"  {msg}")
        summary.append((name, msg))

    print("\n── Summary ──")
    for name, msg in summary:
        print(f"  {name}: {msg}")
    return summary

## Test on a single series

Run this first to verify SAM2 is working before kicking off the full batch.

In [ ]:
# Pick one series to test — change date/dog as needed
TEST_FOLDER = DICOM_ROOT / "20240709" / "GRE2D_FATWATER_WAYLON_0012"

mask_vol, mask_all, rep = segment_series_sam2(TEST_FOLDER, predictor)
print(rep)

## QC: visualise a cross-section of slices

In [ ]:
# Show every 5th slice so we can see propagation quality at the ends
imgs_test = load_dicom_images(str(TEST_FOLDER))
echo = 0

slices_to_show = list(range(0, 50, 5))
fig, axes = plt.subplots(2, len(slices_to_show), figsize=(3 * len(slices_to_show), 6))

for col, s in enumerate(slices_to_show):
    img = imgs_test[echo, s]
    mask = mask_vol[s]

    axes[0, col].imshow(img, cmap="gray")
    axes[0, col].set_title(f"slice {s}")
    axes[0, col].axis("off")

    axes[1, col].imshow(img, cmap="gray")
    if mask.any():
        axes[1, col].imshow(mask, alpha=0.45, cmap="Reds")
    axes[1, col].axis("off")

axes[0, 0].set_ylabel("image", fontsize=9)
axes[1, 0].set_ylabel("overlay", fontsize=9)
plt.suptitle(f"SAM2 propagation — {TEST_FOLDER.parent.name}/{TEST_FOLDER.name}", y=1.01)
plt.tight_layout()
out_png = PROJECT_ROOT / f"{TEST_FOLDER.parent.name}_{TEST_FOLDER.name}_qc.png"
fig.savefig(out_png, dpi=150, bbox_inches="tight")
print(f"Saved: {out_png}")
plt.show()

# Coverage summary
covered = [s for s in range(50) if mask_vol[s].any()]
print(f"Covered slices ({len(covered)}/50): {covered[0]}–{covered[-1]}")

## Compare SAM2 vs SAM1 masks (if SAM1 masks already exist)

In [ ]:
sam1_mask_path = PROJECT_ROOT / "masks_out" / "20240709_GRE2D_FATWATER_WAYLON_0012_mask.npy"

if sam1_mask_path.exists():
    sam1_masks = np.load(sam1_mask_path)
    echo = 0

    # Pick slices where SAM1 failed (empty mask) but SAM2 might have succeeded
    sam1_empty = [s for s in range(50) if not sam1_masks[echo, s].any()]
    print(f"SAM1 failed on {len(sam1_empty)} slices: {sam1_empty}")

    show = sam1_empty[:6] if sam1_empty else list(range(0, 12, 2))
    fig, axes = plt.subplots(3, len(show), figsize=(3 * len(show), 9))

    for col, s in enumerate(show):
        img = imgs_test[echo, s]

        axes[0, col].imshow(img, cmap="gray")
        axes[0, col].set_title(f"slice {s}")
        axes[0, col].axis("off")

        axes[1, col].imshow(img, cmap="gray")
        axes[1, col].imshow(sam1_masks[echo, s], alpha=0.45, cmap="Blues")
        axes[1, col].axis("off")

        axes[2, col].imshow(img, cmap="gray")
        axes[2, col].imshow(mask_vol[s], alpha=0.45, cmap="Reds")
        axes[2, col].axis("off")

    axes[0, 0].set_ylabel("image", fontsize=9)
    axes[1, 0].set_ylabel("SAM1", fontsize=9)
    axes[2, 0].set_ylabel("SAM2", fontsize=9)
    plt.suptitle("SAM1 (blue) vs SAM2 (red) on previously-failing slices", y=1.01)
    plt.tight_layout()
    out_png = PROJECT_ROOT / f"{TEST_FOLDER.parent.name}_{TEST_FOLDER.name}_comparison.png"
    fig.savefig(out_png, dpi=150, bbox_inches="tight")
    print(f"Saved: {out_png}")
    plt.show()
else:
    print("SAM1 mask not found — skipping comparison")

## Full batch run

Once the single-series test looks good, run all 16 series.  
Masks are saved to `masks_out`.

In [ ]:
summary = process_all_sam2(DICOM_ROOT, predictor, OUT_DIR, slice_ranges=SLICE_RANGES)

In [ ]:
# ── One-off: 20240925 Waylon (slices stored in reverse order) ─────────────────
# Slices are flipped relative to all other series.  reverse_slices=True corrects
# this before SAM2 sees the frames.  Update slice_range to match the reversed
# indices after visual QC.

_REV_FOLDER = DICOM_ROOT / "20240925" / "GRE2D_FATWATER_WAYLON_0012"
_REV_NAME = "20240925_GRE2D_FATWATER_WAYLON_0012"
_REV_RANGE = (12, 40)  # adjust as needed

mask_vol_rev, mask_all_rev, rep_rev = segment_series_sam2(
    _REV_FOLDER, predictor,
    slice_range=_REV_RANGE,
    reverse_slices=True,
)
print(rep_rev)

np.save(OUT_DIR / f"{_REV_NAME}_mask.npy", mask_all_rev)
print(f"Saved → {OUT_DIR / (_REV_NAME + '_mask.npy')}")